In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from sys import prefix
from pandas import DataFrame, Series
mp = MasterParams(verbose=True)
io = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import spotify
mio   = spotify.MusicDBIO(verbose=False)
apiio = spotify.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Spotify")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Spotify DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Spotify


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localTracks        = MusicDBData(path=permDir, fname="{0}SearchedForLocalTracks".format(db.lower()))
localTracksData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalTracksData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistIDs     = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDs".format(db.lower()))
localArtistIDsData = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDsData".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Tracks:              {0}".format(len(localTracks.get())))
print("   Local Tracks Data:         {0}".format(len(localTracksData.get())))
print("   Local Artist IDs:          {0}".format(len(localArtistIDs.get())))
print("   Local Artist IDs Data:     {0}".format(len(localArtistIDsData.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

Spotify Search Results
   Global Master Search:      303554
   Global Master Search Data: 0
   Local Artist Search:       530953
   Local Tracks:              217704
   Local Tracks Data:         217704
   Local Artist IDs:          43558
   Local Artist IDs Data:     0
   Local Album Search:        560124
   Errors:                    231
   Known Summary IDs:         1632818


# Download Artist Search Data

In [ ]:
mio   = spotify.MusicDBIO(verbose=False)
apiio = spotify.RawAPIData(debug=True)

## Find Artists To Download

In [ ]:
useMasterNames = False
useChartNames  = True

if useMasterNames is True:
    from musicdb import MusicDBIO
    from pandas import Series
    mdbio = MusicDBIO()
    mmeDF = mdbio.getData()
    unmatchedArtistNames = mmeDF[mmeDF["Spotify"].isna()]["ArtistName"]
    searchedForArtistsSeries = Series(masterArtists.get())
    artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForArtistsSeries.index)]

    print("{0} Search Results".format(db))
    print("   Known Master Artist Names: {0}".format(len(unmatchedArtistNames)))
    print("   Known Local Artist Names:  {0}".format(len(searchedForArtistsSeries)))
    print("   Artist Names To Get:       {0}".format(len(artistNamesToGet)))
elif useChartNames is True:
    searchDF = spotify.MusicDBIO(verbose=False,local=False).data.getSearchArtistData()
    trackArtists = {}
    for trackID,trackData in localTracksData.get().iteritems():
        trackArtists.update({artist['sid']: artist['name'] for artist in trackData["artists"]})
    sTrackArtists = Series(trackArtists)
    artistNamesToGet = sTrackArtists[~sTrackArtists.index.isin(searchDF.index)]

    print("{0} Search Results".format(db))
    print("   Known Master Artist Names: {0}".format(searchDF.shape[0]))
    print("   Track Artist Names:        {0}".format(len(sTrackArtists)))
    print("   Artist Names To Get:       {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:       528927
#   Artist Names To Get:       495346
#   Artist Names To Get:       480868

## Download Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "8:00pm")
#tt = TermTime("today", "2:35pm")
n  = 0
searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if searchedForMasterArtists.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForMasterArtists[artistName] = True
        apiio.sleep(3.5)
        continue
        
    searchedForMasterArtistsData[artistName] = response
    searchedForMasterArtists[artistName] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = DataFrame(getFlatList([v for k,v in mad.items()]))
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = spotify.MusicDBIO(local=False).data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF.index.isin(knownArtists.index).sum()))
    print("  ==> {0} New Artists".format(searchDF.shape[0] - searchDF.index.isin(knownArtists.index).sum()))
    print("Saving Data")
    spotify.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
masterArtistsData.save(data={})

# Download Data By Track IDs (only for Spotify Charts)

In [ ]:
mio   = spotify.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = spotify.RawAPIData(debug=True)

In [ ]:
from pandas import read_csv
df = read_csv("/Volumes/Piggy/Charts/data/spotify/full/charts.csv")
data = df["url"].unique()
print(len(data))
chunks = [data[x:x+50] for x in range(0, len(data), 50)]

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
localTracksDict = localTracks.get()
for i,tracks in enumerate(chunks[4150:]):
    tracksDict = {item.split("/")[-1]: item for item in tracks}
    tracksDict = {trackID: track for trackID,track in tracksDict.items() if localTracksDict.get(trackID) is None}
    print(i,'\t',len(tracksDict))
    if len(tracksDict) == 0:
        continue
    response = apiio.getTracksLookupResults(tracks=tracksDict.values())
    if len(response) > 0:
        localTracksDict.update({item['sid']: item for item in response})
    apiio.sleep(7.5)
    n += 1
        
    if n % 10 == 0:
        print("="*150)
        ts.update(n=n, N=len(chunks))
        print("Saving {0} {1} Searched For Track IDs".format(len(localTracksDict), db))
        localTracks.save(data=localTracksDict)   
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break

print("Saving {0} {1} Searched For Track IDs".format(len(localTracksDict), db))
localTracks.save(data=localTracksDict) 

In [ ]:
localTracksData.save(data=localTracks.get())

In [ ]:
len(trackArtists)

In [ ]:
localTracks.save(data=tmp)

# Download Data By Artist IDs (generally can be handled by 'Artist Search')

In [ ]:
mio   = spotify.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = spotify.RawAPIData(debug=True)

## Find Artists To Download

In [ ]:
useMissingArtists = False
useSearchResults  = False
useTracksSearches = False
useMasterIDs      = True

if useMasterIDs is True:
    import musicdb
    pdbio = musicdb.MusicDBIO()
    mmeDF = pdbio.getData()
    spotData = mmeDF[mmeDF["Spotify"].notna()][["Spotify", "ArtistName"]]
    artistNames = {}
    for idx,row in spotData.iterrows():
        artistIDs = row["Spotify"]
        artistName = row["ArtistName"]
        if isinstance(artistIDs,list):
            for artistID in artistIDs:
                if artistID == "https":
                    print(idx)
                artistNames[artistID] = artistName
        else:
            artistNames[artistIDs] = artistName
    artistNames = Series(artistNames)
    print("   Possible Searched IDs:   {0}".format(len(artistNames)))
    print("   Downloaded Searched IDs: {0}".format(len(knownArtists)))
    artistIDsToGet = artistNames[~artistNames.index.isin(knownArtists.keys())]
    print("   Artists IDs To Get:      {0}".format(len(artistIDsToGet)))
if useTracksSearches is True:
    tracksArtistsData = {}
    for trackID,trackData in localTracksData.get().iteritems():
         for artistData in trackData['artists']:
                artistID   = artistData['sid']
                artistName = artistData['name']
                if tracksArtistsData.get(artistID) is None:
                    tracksArtistsData[artistID] = {"N": 0, "Name": artistName}
                tracksArtistsData[artistID]["N"] += 1
    artistNames = DataFrame(tracksArtistsData).T.sort_values(by="N", ascending=False)["Name"]    
    print("   Possible Searched IDs:   {0}".format(len(artistNames)))
    print("   Downloaded Searched IDs: {0}".format(len(knownArtists)))
    artistIDsToGet = artistNames[~artistNames.index.isin(knownArtists.keys())]
    print("   Artists IDs To Get:      {0}".format(len(artistIDsToGet)))
if useSearchResults is True:
    print("{0} Search Results".format(db))
    artistNames = spotify.MusicDBIO(local=False).data.getSearchArtistData()['name']
    print("   Possible Searched IDs:   {0}".format(len(artistNames)))
    localArtistIDsDict = localArtistIDs.get()
    print("   Downloaded Searched IDs: {0}".format(len(localArtistIDsDict)))
    artistIDsToGet = artistNames[~artistNames.isin(localArtistIDs.get().keys())]
    print("   Artists IDs To Get:      {0}".format(len(artistIDsToGet)))
elif useMissingArtists is True:
    print("{0} Search Results".format(db))
    print("   Known Summary IDs:    {0}".format(len(knownArtists)))
    artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
    print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
    artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(localArtistIDs.get().keys())]
    print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

In [ ]:
spotData = mmeDF[mmeDF["Spotify"].notna()][["Spotify", "ArtistName"]]
for idx,row in spotData.iterrows():
    artistIDs = row["Spotify"]
    artistName = row["ArtistName"]
    if isinstance(artistIDs,list):
        for artistID in artistIDs:
            if artistID in artistIDsToGet.index:
                print('pdbio.setdbid(\"{0}\", \"Spotify\", \"\")  ## {1} / {2}'.format(idx, artistIDs, artistName))
    else:
        if artistIDs in artistIDsToGet.index:
            print('pdbio.setdbid(\"{0}\", \"Spotify\", \"\")  ## {1} / {2}'.format(idx, artistIDs, artistName))

## Download Artist Info Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = localArtistIDsData.get()
searchedForArtistIDs     = localArtistIDs.get()
searchedForErrors        = errors.get()
for i,(dbID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForArtistIDs.get(dbID) is not None:
        print("Known ==> {0}".format((dbID,artistName)))
        continue
    if searchedForErrors.get(dbID) is not None:
        print("Error ==> {0}".format((dbID,artistName)))
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID, artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        localArtistIDs.save(data=searchedForArtistIDs)
        localArtistIDsData.save(data=searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
if True:
    print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
    localArtistIDs.save(data=searchedForArtistIDs)
    localArtistIDsData.save(data=searchedForArtistIDsData)
    if len(searchedForErrors) > 0:
        print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
        errors.save(data=searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

# Download Album Data

In [6]:
mio   = spotify.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = spotify.RawAPIData(debug=True)

Spotify API(Key={'client_id': '61e441c3b90c4873aa0e6b9582564f95', 'client_secret': 'ae0d0f968bf443fdac1d9ac6ef65fc0f'})


## Find Albums To Get

In [58]:
print("# {0} Search Results".format(db))
print("#   Known Summary IDs:         {0}".format(len(knownArtists)))
searchedForAlbums = localAlbums.get()
print("#   Download Artist Album IDs: {0}".format(len(searchedForAlbums)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(localAlbums.get().keys())]
print("#   Artists IDs To Get:        {0}".format(len(artistNamesToGet)))

#   Artists IDs To Get:        955919

# Spotify Search Results
#   Known Summary IDs:         1632818
#   Download Artist Album IDs: 692479
#   Artists IDs To Get:        940339


## Download Albums Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "8:00pm")
n  = 0
searchedForAlbums = localAlbums.get()
searchedForErrors = errors.get()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(5)
    n += 1
        
    if n % 10 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        localAlbums.save(data=searchedForAlbums)
        print("="*150)
        apiio.wait(7)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.wait(15)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
localAlbums.save(data=searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Spotify Albums] Start    ==> Time Is 2022-05-23 07:07:37
========================= termTime(day=today,time=8:00pm) =========================
   ====> Terminate Time Set To 2022-05-23 20:00:00 <====
   ====> Will Terminate Process 12 Hours and 52 Minutes From Now
Searching For Albums For Sjuwana Byers & Children of God (4oJyx5av6yHBPZv5yzPFOM)	   ===> [3]        3  3
Searching For Albums For Da Daze (6Mcla59Uf6HWDdUIwRGzMH)                  	   ===> [20]       20  20
Searching For Albums For Dale and Diane Hunter (3lhQlGugjil8ueUjrWLL5J)    	   ===> [1]        1  1
Searching For Albums For Kansas City Repertory Singers (46mZE1pvuObQMAMuVw7PMN)	   ===> [1]        1  1
Searching For Albums For Bounty Killer With Chuck Turner (0KMxoIe3ISTy30HQq7y6qS)	   ===> [2]        2  2
Searching For Albums For Njayokhokho (5ceQvNNoSXDMCrVKH90K1u)              	   ===> [5]        5  5
Searching For Albums For Reverend Milton Brunson (1vq6UfGGTw0XOfGKmsWFWm)  	   ===> [2]        2  2
Se

Waiting:   0%|          | 0/70 [00:00<?, ?it/s]

Searching For Albums For My Favorite Citizen (7pxpeGQj33Uc1GdgXwj1IH)      	   ===> [1]        1  1
Searching For Albums For SALEM (0bRoJvURu4btP380hswmSv)                    	   ===> [1]        1  1
Searching For Albums For Nguyen Ngoc Ngan (4bLgC7MIO7hkwipgCv71jR)         	   ===> [2]        2  2
Searching For Albums For Junior Mance and the FJF Trio with special guests Arturo Sandoval, Etta Jones, and Lou Donaldson (5ewiUFvBntZOQIfac8GU2I)	   ===> [1]        1  1
Searching For Albums For OMEGA TRIBE (2yMhtZBdiyuMfuaX3jBUt4)              	   ===> [2]        2  2
Searching For Albums For Kenneth Jones (7252ilADS0RgGZxyF9SuaP)            	   ===> [38]       38  38
Searching For Albums For Gravel Road Acoustic Trio (2TZHT5KlOTkvi4SB3SkQNQ)	   ===> [1]        1  1
Searching For Albums For Petya Ivanova (2w5MYhpBu0krx6lMUQxNxg)            	   ===> [4]        4  4
Searching For Albums For Anna Hill (7oHD23BgsiXfPjEUsXc9HM)                	   ===> [1]        1  1
Searching For Albums For Da

Waiting:   0%|          | 0/70 [00:00<?, ?it/s]

Searching For Albums For D. Savage (2Z0q4EyLb1jVi082o4RUJh)                	   ===> [3]        3  3
Searching For Albums For Rhiannon Llewellyn (1tMnvnkQbWH4Ya23BVSsaR)       	   ===> [2]        2  2
Searching For Albums For Benz Nct (2ElC9QxJie0XKtuhx0xlbP)                 	   ===> [4]        4  4
Searching For Albums For Martin Breinschmid & The Radio Kings (5NXHqShVsGj4bJ1TtZHPCU)	   ===> [4]        4  4
Searching For Albums For Cardan Balam (4o670tCUXXhMKuz2leE0ob)             	   ===> [1]        1  1


...

Searching For Albums For Dafick (4d7qX2Bm6zVZ1emQ89BwU3)                   	   ===> [1]        1  1
Searching For Albums For Conflicted (0XeVc0KzaVTDvxiGA89Gtt)               	   ===> [1]        1  1
Searching For Albums For Paolo Polifrone (50GQiylAsEz3oUDhoW8RNW)          	   ===> [1]        1  1
Searching For Albums For DIM4OU (5l68Jt38hKrKhwdIZB43xu)                   	   ===> [1]        1  1
Searching For Albums For Hugo Wolf Quartett (4OlheOvl5XfnEuvxAkJ8B8)       	   ===> [1]        1  1
2530/?     : Process [Getting Spotify Albums] Has Run For 5 Hours and 1 Minute.
Saving 695009 Spotify Searched For Albums


Waiting:   0%|          | 0/70 [00:00<?, ?it/s]

Searching For Albums For Adel Salameh & Eduardo Niebla (51aQ7uXepvDORSzlIqjziH)	   ===> [1]        1  1
Searching For Albums For Roke Dj (39PkdaVf2Fi1Fm1UBzKTXs)                  	   ===> [8]        8  8
Searching For Albums For Darian Williams (2Z66XnM5sBCwNyBV0xJ71F)          	   ===> [1]        1  1
Searching For Albums For Karl Priore (1bXfeCJq7yGLU2QMKSMSFG)              	   ===> [1]        1  1
Searching For Albums For El Wafir (1XxzicMfEFjTMw4sheyTfZ)                 	   ===> [1]        1  1
Searching For Albums For Mimitah (7Dcr4PpwDoSME8fJtjeWBP)                  	   ===> [2]        2  2
Searching For Albums For Timberwolf (3k32rgWU2OnhTeIHc5YSlF)               	   ===> [3]        3  3
Searching For Albums For Wicked Intervals (7m7EosNZCR82VeLB3shle3)         	   ===> [2]        2  2
Searching For Albums For Wafiq (0uux30Ly2eDtxxWwmZJvcF)                    	   ===> [2]        2  2
Searching For Albums For Cari Lekebusch & Joel Mull (55feI7bWzjCZivp0c6vKrz)	   ===> [1]        

Waiting:   0%|          | 0/70 [00:00<?, ?it/s]

Searching For Albums For Hmu Hmu Khin (0mhTE1CKZM417xzhO7ANlA)             	   ===> [2]        2  2
Searching For Albums For Lil Jay Wop (0h9O0iHGli7PCv7mpUK2fq)              	   ===> [1]        1  1
Searching For Albums For Darina (1Q1y3WWLu0APPEtF83dcPn)                   	   ===> [1]        1  1
Searching For Albums For Steve Bakken (2H5bK7TboaaCzn69pCbFEZ)             	   ===> [1]        1  1
Searching For Albums For Despina Olympiou (34viOK6uod0NydAEIid391)         	   ===> [2]        2  2
Searching For Albums For Henry G. Neubert (46gGZrbmhBHtRg3HmmJKtw)         	   ===> [1]        1  1
Searching For Albums For Kara Lodger (47NCU1WTGFkhiRDofcwACp)              	   ===> [3]        3  3


## Move Local Files

In [ ]:
from lib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
from utils import PoolIO
pio = PoolIO("Spotify")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Download Artist Info Data

## Determine Artists To Download

## Download Artist Info

# Download Artist Albums Data

## Determine Artists To Download

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

In [ ]:
searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = spotifyArtists[~spotifyArtists.index.isin(searchedForArtists.index)]['name']
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
from pandas import Series
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    sids = [sid for sid in sids if len(sid) == 22]
    spotifyToGet.update({sid: name for sid in sids})
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Spotify")
api  = apig.getDBAPIData("Spotify")

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
ts = timestat("Downloading Spotify Data")
#tt = termTime("today", "10:00pm")
tt = termTime("tomorrow", "10:00pm")
#tt = termTime("tomorrow", "7:00am")
#tt = termTime("today", "9:30am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistAlbumsKnown(dbID) or True:
        response = api.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistAlbums(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        print("="*150)
        api.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        api.sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

# Extra

In [ ]:
  
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)

In [ ]:
######################################################
# Juypter
######################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from ioUtils import getFile, saveFile
from webUtils import getHTML
from searchUtils import findExt
from fsUtils import setFile, isFile, fileUtil
from fileUtils import getBaseFilename
from timeUtils import timestat
from fileIO import fileIO
from dbUtils import utilsSpotifyCharts


###########################################################################
## General
###########################################################################
import urllib
from glob import glob
from time import sleep
from io import StringIO
from pandas import Series,DataFrame,concat,read_csv,date_range
from datetime import datetime
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath



######################################################
# Versions
######################################################
import sys
print("Python: {0}".format(sys.version))
import datetime as dt
start = dt.datetime.now()

print("Notebook Last Run Initiated: "+str(start))

# Global

In [ ]:
io = fileIO()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()

from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

# Spotify API

In [ ]:
# !pip install spotipy

In [ ]:
from artistSpotify import artistSpotify
#from artistIDBase import artistIDSpotify
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection


##################################################################################################################
# Base Class
##################################################################################################################
class dbSpotify:
    def __init__(self):
        self.db     = "Spotify"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistSpotify(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        self.manc = masterArtistNameCorrection()
        
        
    def getSearchDir(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return searchDir
        
    def getArtistSearchSavename(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return fileUtil(searchDir).join("{0}.p".format(psID))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
def getArtistTrackResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
            sleep(1.0)
    print(" {0}".format(len(searchResults)))
    return searchResults

In [ ]:
def getArtistAlbumResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    href           = requestResult.get('href')
    artistID       = PurePosixPath(unquote(urlparse(href).path)).parts[3]
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        sleep(3.5)
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
    print(" {0}".format(len(searchResults)))
    
    albums = [spotifyAlbumRecord(album).get() for album in searchResults]
    retval = {'href': href, 'artistID': artistID, 'albums': albums}
    return retval

In [ ]:
def getArtistSearchResults(artistName, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    sleep(0.25)
    result = sp.search(artistName, limit=limit, type='artist')
    
    artists = result.get('artists', {}) if isinstance(result,dict) else {}
    href    = artists.get('href')
    total   = artists.get('total')
    nextURL = artists.get('next')
    prevURL = artists.get('previous')
    items   = artists.get('items', [])
    retval  = [spotifyArtistRecord(item).get() for item in items]
    print(len(retval))
    
    return retval

# Artist Search

In [ ]:
from glob import glob
from masterUtils import masterUtils
mu = masterUtils()
disc = mu.getDisc("Spotify")
ts = timestat("Finding Search Files")
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
ts.stop()

In [ ]:
from fileIO import fileIO
io = fileIO()
#io.save(idata=files, ifile="spotifyFiles.p")
files = io.get("spotifyFiles.p")
len(files)

In [ ]:
from fsUtils import fileUtil, mkDir, dirUtil, moveFile
dbIO = dbSpotify()
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    artistName = fileUtil(ifile).basename
    searchDir = dbIO.getSearchDir(artistName)
    if not searchDir.exists():
        print("Making {0}".format(searchDir))
        mkDir(searchDir)
    savename = dbIO.getArtistSearchSavename(artistName=artistName)
    moveFile(ifile,savename)
    
    if (i+1) % 10000 == 0 or (i+1) == 1000 or (i+1) == 100:
        ts.update(n=i+1, N=len(files))
ts.stop()

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
ts = timestat("Getting Searched For Artists")
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
ts = timestat("Getting Artists To Download")
mmeDF = meu.getDF()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtists)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 2500:
        break
    savefile = fileUtil("spotify/artists").join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if savefile.exists():
        continue
    if nErr >= 5:
        break
    result = getArtistSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(2.5)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
    
    if (i+1) % 100 == 0:
        sleep(120)
ts.stop()

## Artist Results

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()

# Move Artist Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile

files = glob("spotify/artists/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    dst = dirUtil("/Volumes/Piggy/Discog/artists-spotify/artists").join(src.name)
    if dst.exists():
        continue
    moveFile(src.path, dst)

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

# Move Album Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile,mkDir
from artistModValue import artistModValue
amv = artistModValue()

files = glob("spotify/albums/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    albumsData = io.get(ifile)
    artistID = albumsData['artistID']
    modValue = amv.getModVal(artistID)
    modDir = dirUtil("/Volumes/Piggy/Discog/artists-spotify/").join(str(modValue))
    if not modDir.exists():
        mkDir(modDir)
    dst = dirUtil(modDir).join("{0}.p".format(artistID))
    if dst.exists():
        continue
    print("  ==>  {0}".format(dst))
    moveFile(src.path, dst)

# Create Master Artist ID List

In [ ]:
from glob import glob
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
artistsData = []
N = len(files)
ts = timestat("Loading {0} Spotify Artist Search Files".format(N))
for i,ifile in enumerate(files):    
    artistsData += io.get(ifile)
    if (i+1) % 5000 == 0 or (i+1) == 1000:
        ts.update(n=i+1, N=N)
ts.stop()

In [ ]:
def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

df = DataFrame(artistsData)
df["followers"] = df["followers"].apply(getFollowers)
df.index = df['sid']
df.drop(['sid'], axis=1, inplace=True)
df = df[~df.index.duplicated(keep='first')]
df = df.sort_values(by='popularity', ascending=False)

print("Saving {0} Spotify Artists Data".format(df.shape[0]))
io.save(idata=df, ifile="/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
df.head(10)

In [ ]:
print(df.shape)

# Artist Albums

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

## Download Albums Data

In [ ]:
spotifyArtists['popularity'].hist(bins=100, log='y')

In [ ]:
spotifyArtists[spotifyArtists["popularity"] >= 60].shape

In [ ]:
knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
spotifyToGet = Series(spotifyToGet)
#spotifyToGet = knownSpotify["ArtistName"]
#spotifyToGet.index = list(knownSpotify["Spotify"].values)
#spotifyToGet.name = "name"
#spotifyToGet

In [ ]:
toget = {}
from artistModValue import artistModValue
from masterUtils import masterUtils
from fsUtils import dirUtil, mkDir
mu   = masterUtils()
amv  = artistModValue()
disc = mu.discs["Spotify"]


#spotifyToGet = spotifyArtists[spotifyArtists["popularity"] >= 60]
#print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
#for i,(artistID,artistName) in enumerate(spotifyToGet["name"].iteritems()):

print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
nKnown = 0
for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
    modVal   = amv.getModVal(artistID)
    modDir   = dirUtil(disc.getArtistsDir()).join(str(modVal))
    if not modDir.exists():
        print("Making {0}".format(modDir))
        mkDir(modDir)
    savefile = dirUtil(modDir).join("{0}.p".format(artistID))
    if savefile.exists():
        nKnown += 1
        continue
    #print("{0: <25}{1: <20} ({2})".format(artistName,artistID,modVal))
    toget[savefile] = (artistName,artistID)
    if len(toget) >= 2500:
        break
print("Found {0} Known Artists".format(nKnown))
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(7.5)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(15.0)
        print("")
    
    if (i+1) % 25 == 0:
        sleep(30.0)
ts.stop()

## Determine Artists To Download

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
io.save(idata={}, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
stop = 150
dbIO = dbSpotify()
api  = spotifyAPIIO()
ts   = timestat("Getting Artist Data From Spotify API")
n    = 0

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    savename = dbIO.getArtistAlbumsSavename(artistID)
    if savename.exists():
        searchedForArtists[artistID] = artistName 
        continue
        
    #print(artistID,artistName,savename)
    response = api.getArtistAlbums(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistAlbums(artistID=artistID, artistAlbums=response)
    searchedForArtists[artistID] = artistName    
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Error Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
if False:
    searchedForArtists = io.get("spotifySearchedForArtists.p")
    print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

    ts = timestat("Finding Spotify Saves")
    first = True
    for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
        if searchedForArtists.get(artistID) is not None:
            continue
        if first:
            print("Start: {0}".format(i))
            first = False
        if dbIO.getArtistAlbumsSavename(artistID).exists():
            searchedForArtists[artistID] = artistName
            if len(searchedForArtists) % 5000 == 0:
                break
        else:
            break

        if (i+1) % 100 == 0:
            ts.update(n=i+1,N=len(spotifyToGet))        
            #io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

    ts.stop()
    print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
    io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

In [ ]:
spotifyArtists.head()

In [ ]:
from dbArtistsBase import dbArtistsBase
from fileUtils import getBaseFilename
from fsUtils import isFile, setFile, setDir
from ioUtils import getFile, saveFile
from timeUtils import timestat
from sys import prefix
import urllib
from time import sleep
from webUtils import getHTML
from pandas import Series
    
from fileIO import fileIO
from fsUtils import fileUtil

In [ ]:
from dbArtistsParse import dbArtistsSpotifyAPI
from dbArtistsSpotify import dbArtistsSpotify
dbAP = dbArtistsSpotifyAPI(dbArtistsSpotify())

In [ ]:
for modVal in range(100):
    dbAP.parse(modVal=modVal, expr='< 0 Days', force=False)
    
#for modVal in range(100):
#    dbAP.parseSearch(modVal, expr='< 0 Days', force=False)

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()
modVal = 91
idx = artistSearchData.reset_index()['sid'].apply(amv.getModVal) == modVal
idx.index = artistSearchData.index
artistSearchData[idx]

In [ ]:
artistSearchData.head()

In [ ]:
art = artistSpotify()
art.getData(inputData).show()

In [ ]:
io.get("/Users/tgadfort/Music/Discog/artists-spotify-db/91-DB.p").get('1uNFoZAHBGtllmzznpCI3s').show()

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
mmeDF = meu.getDF()

In [ ]:

toget = {}
for i,(artistID,artistName) in enumerate(spotifyArtists["name"].iteritems()):
    if i >= 5:
        break
    savefile = fileUtil("spotify/albums").join("{0}--{1}.p".format(artistID,manc.clean(artistName)))
    if savefile.exists():
        pass
        #continue
    toget[savefile] = (artistName,artistID)
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(3.0)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

## Artist Albums Results

In [ ]:
from glob import glob
files = glob("spotify/albums/*.p")
artistAlbumsData = {}
for ifile in files:
    albumsData   = io.get(ifile)
    artistID     = albumsData['artistID']
    artistAlbums = albumsData['albums']
    if artistAlbumsData.get(artistID) is not None:
        raise ValueError("Already have data for {0}".format(artistID))
    artistAlbumsData[artistID] = artistAlbums

In [ ]:
retval = getArtistAlbumResults(artistID='46sPgFJpEcoxUJNr1ouR9G', artistName='dummy')

In [ ]:
s = Series(artistAlbumsData)

In [ ]:
s.apply(len)

In [ ]:
retval = sp.artist_albums('13y7CgLHjMVRMDqxdx0Xdo', limit=10, offset=0)

In [ ]:
retval['href']

In [ ]:
href='https://api.spotify.com/v1/artists/13y7CgLHjMVRMDqxdx0Xdo/albums?offset=0&limit=10&include_groups=album,single,compilation,appears_on'
href

In [ ]:
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath

url = 'http://www.example.com/hithere/something/else'

PurePosixPath(unquote(urlparse(href).path)).parts[3]

In [ ]:
from urlparse import urlparse, parse_qs
s = "https://xx.com/question/index?qid=2ss2830AA38Wng"
parse_qs(urlparse(s).query)['qid'][0]

In [ ]:
rParen = r'\((.*?)\)+'
rFeat  = r'\b(feat.\s|Feat.\s|with\s)\b'
rSuffix= r'\s-\sRemix'

In [ ]:
featureValue = regex.search(rFeat, parenValue.group())


In [ ]:
retval = sp.artist_top_tracks('60d24wfXkVzDSfLS6hyCjZ')

In [ ]:
help(sp.tracks)

In [ ]:
requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)


In [ ]:
DataFrame(retval['items']).T[0]

In [ ]:
retval2 = sp.artist_albums('60d24wfXkVzDSfLS6hyCjZ', limit=50, offset=450)

In [ ]:
retval2['previous']

In [ ]:

results = sp.search(q='weezer', limit=20)
for idx, track in enumerate(results['tracks']['items']):
    print(idx, track['name'])

In [ ]:
result.keys()

In [ ]:
urn = 'spotify:artist:3jOstUTkEu2JkjvRdBA5Gu'

#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())

artist = sp.artist(urn)
artist

In [ ]:
result['artists'].keys()

In [ ]:
help(sp.search)

In [ ]:
search_str = 'Radiohead'
result = sp.search(search_str)
pprint.pprint(result)